In [ ]:
# Importing libraries/ packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from collections import Counter
from math import sqrt

from sklearn.metrics import accuracy_score

In [ ]:
# Reading dataset
df = pd.read_csv(r"train.csv")
df.head()

In [ ]:
# Checking shape 
df.shape

# Data Preprocessing

In [ ]:
# Removing PassengerId, Name and Ticket Columns from the analysis \
# as they do not add any learning ability to the problem statement
df = df.drop(['PassengerId','Name','Ticket'], axis = 1)
df.shape

In [ ]:
# Encoding Categorical variable as numbers
label_encoder = preprocessing.LabelEncoder()

df['Sex']= label_encoder.fit_transform(df['Sex'])
df['Embarked']= label_encoder.fit_transform(df['Embarked'])
df.head()

In [ ]:
# Check DataType 
df.dtypes

In [ ]:
# Convert datatype of Categorical columns 
df.Pclass = df.Pclass.astype('object')
df.SibSp = df.SibSp.astype('object')
df.Parch = df.Parch.astype('object')
df.dtypes

In [ ]:
# Check presence of NULLs
df.isnull().sum()

In [ ]:
# Check number of categories in 'Cabin' column
df.Cabin.nunique()

In [ ]:
# Percentage of nulls in Cabin and Age column
print(np.round((len(df[df.Cabin.isnull()])/ len(df) * 100),2),"  % of records in CABIN Column are NULL")
print(np.round((len(df[df.Age.isnull()])/ len(df) * 100),2)," % of records in AGE Column are NULL")
print(np.round((len(df[df.Embarked.isnull()])/ len(df) * 100),2)," % of records in EMBARKED Column are NULL")

In [ ]:
# CABIN Column
# Due to nearly 77% of values in CABIN column being NULL and the column having 147 unique categories, \
# it would not be possible to impute/treat null values. 
# Hence 'CABIN' column is removed from the analysis
df = df.drop('Cabin', axis = 1)

# AGE Column
# Impute NULL values in AGE Column using Mean Age
df['Age']=df['Age'].fillna(df['Age'].mean())

# EMBARKED Column
# Since only 0.22% of the records are NULL, NULL records are removed from the analysis
df = df[df.Embarked.notnull()]

In [ ]:
# Check Class Imbalance
print(df.Survived.value_counts())
ax = df.Survived.value_counts().plot(kind='bar', title='Check Class imbalance in \'Survived\' column')
ax.set_xlabel("Survived or Not Survived")
ax.set_ylabel("Number of Records")

In [ ]:
# Since not much imbalance exists between 0 (Not-Survived) and 1 (Survived)
# Hence proceeding ahead

In [ ]:
# Scaling to bring numerical column values to same scale using MinMaxScaler
df_numerical = df[['Age','Fare']]
scaler = MinMaxScaler()
scaled_df_numerical = scaler.fit_transform(df_numerical)
df_numerical = pd.DataFrame(np.round(scaled_df_numerical,2), columns=['Age','Fare'])
df['Age'] = df_numerical['Age']
df['Fare'] = df_numerical['Fare']
df.head()

In [ ]:
# Split Train and Test dataset for validating Machine Learning model
y = df.Survived
x = df.drop('Survived', axis=1)
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.2, random_state = 42)
train_x.shape, test_x.shape, train_y.shape, test_y.shape

# Model Building

# KNN

In [ ]:
# Calculate the Euclidean distance between two vectors
def euclidean_distance(r1, r2):
    
    distance = 0
    for i in range(len(r1)-1):
        distance += (r1[i] - r2[i])**2
    distance = distance**(1/2)
    return distance

In [ ]:
# KNN Model 
def knn_predict(train_x, test_x, k):

    prediction_y = []
    train_x = train_x.iloc[:,:-1].values
    test_x = test_x.iloc[:,:-1].values

    for test_pt in test_x:
        distances = []

        for train_pt in train_x:
#             For a selected test point, find Euclidean distance between that test point and train point
#             print(test_pt, train_pt)
            distance = euclidean_distance(test_pt, train_pt)
            distances.append(distance)
        
        df_distances = pd.DataFrame(distances, columns=['distances'], index=train_y.index)
        
        # Sort distances, consider k closest points
        df_nearest_neighbors = df_distances.sort_values(by=['distances'], axis=0)[:k]

        # Counter to track the labels of k closest neighbors
        counter = Counter(train_y[df_nearest_neighbors.index])

        # Fetch most common label/class of all the nearest neighbors
        prediction = counter.most_common()[0][0]
        
        # Append prediction to output list
        prediction_y.append(prediction)
        
    return prediction_y

In [ ]:
# Predictions
prediction_y = knn_predict(train_x, test_x, k=5)

# Accuracy
print(accuracy_score(test_y, prediction_y))

In [ ]:
# Choosing value of k 
range_k = range(1, 10)
accuracy_scores = []
for i in range_k:
    prediction_y = knn_predict(train_x, test_x, k=i)
    accuracy_scores.append(accuracy_score(test_y, prediction_y))
    
df_k_selection = pd.DataFrame(range_k, columns=['k_value'])
df_k_selection['Accuracy_Score'] = accuracy_scores
df_k_selection

In [ ]:
# from above results, optimal value of k is 7 (beyond which accuracy saturates to a constant)
optimal_k = 7

In [ ]:
# Final Model
prediction_y = knn_predict(train_x, test_x, k=optimal_k)

# Accuracy
print(accuracy_score(test_y, prediction_y))

In [ ]:
# Run model on test data
# Reading dataset
test_df = pd.read_csv(r"test.csv")
test_df.head()

In [ ]:
# Data Preprocessing
# Removing PassengerId, Name and Ticket Columns from the analysis 
test_df = test_df.drop(['PassengerId','Name','Ticket'], axis = 1)

# Label encoding
test_df['Sex']= label_encoder.fit_transform(test_df['Sex'])
test_df['Embarked']= label_encoder.fit_transform(test_df['Embarked'])

# Convert datatype of Categorical columns 
test_df.Pclass = test_df.Pclass.astype('object')
test_df.SibSp = test_df.SibSp.astype('object')
test_df.Parch = test_df.Parch.astype('object')

# Dropping Cabin column
test_df = test_df.drop('Cabin', axis = 1)

# AGE Column
# Impute NULL values in AGE Column using Mean Age
test_df['Age']=test_df['Age'].fillna(test_df['Age'].mean())

# EMBARKED Column
# Remove NULL
test_df = test_df[test_df.Embarked.notnull()]

# Scaling
df_numerical = test_df[['Age','Fare']]
scaler = MinMaxScaler()
scaled_df_numerical = scaler.fit_transform(df_numerical)
df_numerical = pd.DataFrame(np.round(scaled_df_numerical,2), columns=['Age','Fare'])
test_df['Age'] = df_numerical['Age']
test_df['Fare'] = df_numerical['Fare']
test_df.head()

In [ ]:
# Predictions on Test data
prediction_y = knn_predict(train_x, test_df, k=optimal_k)

In [ ]:
# Reading test_y
test_y = pd.read_csv(r"gender_submission.csv")
test_y.head()

In [ ]:
# Comparing test results using accuracy score
test_y = test_y.Survived
print(accuracy_score(test_y, prediction_y))